# Шаг 1в: полный перевод запросов CodeSearchNet моделью NLLB-600M (EN→RU, RU→EN)

Переводчик выбран слепым сравнением (`results/step1b-report.md`): NLLB-600M, μ=3,36
против 2,67 у opus-mt, p<0,0001. На CPU NLLB переводит 0,086 запроса/с, поэтому полный
прогон 100k выполняется здесь, на GPU Colab (T4); оценка 3–5 ч.

## Что загрузить (два файла из проекта)

| файл | из папки | размер |
|---|---|---|
| `semsearch.zip` — код пакета | `kursovaya-tex/code/semsearch/` | ~30 КБ |
| `csn_prepared.tar.gz` — данные | `kursovaya-tex/code/var/data/*.jsonl` | ~50–70 МБ |

```bash
# из корня проекта study-ensemble
cd kursovaya-tex/code
zip -r ../notebooks/semsearch.zip semsearch
tar czf ../notebooks/csn_prepared.tar.gz -C var/data train.jsonl val_queries.jsonl test_queries.jsonl
ls -lh ../notebooks/semsearch.zip ../notebooks/csn_prepared.tar.gz
```

## Как запускать

1. https://colab.research.google.com/ → **File → Upload notebook** → `translate_nllb.ipynb`.
2. **Runtime → Change runtime type → T4 GPU** → Save.
3. Запустить ячейку «Загрузка файлов в Colab» — в диалоге выбрать **оба** файла сразу.
   Если файлы загружены, а ячейка ещё ждёт — выбрать их снова; это не ошибка.
4. **Runtime → Run all**. Порядок: окружение → GPU → распаковка → модель → train EN→RU →
   train RU→EN → val/test → отчёт. Первое скачивание модели NLLB (~2,5 ГБ, 1–2 мин),
   затем 6 прогонов перевода.
5. **Сохранение работы.** Бесплатная сессия Colab живёт 3–4 ч, а в конце ячейка
   «Сохранить результаты» скачивает `nllb_translation_results.zip` (кэш переводов +
   готовые jsonl + отчёт). Если не успеваете — запустить эту ячейку вручную в любой момент
   (она стоит после основной цепочки, её можно запустить, пока остальные работают),
   скачать архив, закрыть сессию, а в следующем сеансе загрузить архив в форме
   «Загрузка файлов»: ноутбук распакует его и продолжит с места обрыва.
6. Архив положить в `kursovaya-tex/results/`; переведённые jsonl (внутри архива,
   `/content/results/`) — вход Шага 2.

## Что делает ноутбук

- `semsearch/` → `/content/semsearch/`, добавить в `sys.path`; `SEMSEARCH_HOME=/content/cache`.
- Вся логика перевода и кэша — в `semsearch/data/translate.py` (`DIRECTIONS`,
  `Seq2SeqBackend`, `translate_file`, `TranslationCache`). Ноутбук только держит загруженную
  модель в памяти (иначе каждый из 6 прогонов заново загружал бы 2,5 ГБ весов) и собирает
  сводку.
- Направления: `nllb-en2ru` — `query_en` → `query_ru`; `nllb-ru2en` — `query_ru` →
  `query_en_back` (обратный перевод для BM25). Результат — копия входного jsonl с новым полем.
- Кэш — `/content/cache/cache/translation/nllb-200-distilled-600M-eng_Latn-rus_Cyrl.jsonl`
  и `...-rus_Cyrl-eng_Latn.jsonl`; форма записи `{"key", "text", "translation"}`, ключ —
  sha256 от модели и текста.
- Сводка — `/content/results/translation_report.json`.

## 1. Установка зависимостей

In [ ]:
import subprocess, sys

# Colab: torch с CUDA уже стоит — ставим только остальное. setuptools обновляем:
# transformers 5.x ищет пакеты через importlib.metadata, а в Colab он старый.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers>=5.0", "sentencepiece", "tqdm", "setuptools"], check=True)

import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__)

## 2. Проверка GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU нет: Runtime → Change runtime type → T4 GPU"
free_gb, total_gb = [x / 1024**3 for x in torch.cuda.mem_get_info()]
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Свободно {free_gb:.1f} ГБ из {total_gb:.1f} ГБ; модель NLLB-600M займёт ~2.5 ГБ")

DEVICE = "cuda"
BATCH_SIZE = 16
NUM_BEAMS = 4

## 3. Загрузка файлов в Colab

Выбрать `csn_prepared.tar.gz` и `semsearch.zip` (или `nllb_translation_results.zip`
при продолжении). Ячейка ждёт выбора файлов до 30 мин, затем распаковывает архивы и
проверяет, что на месте нужные jsonl и код. Пути задаются здесь: `DATA`, `SEMSEARCH`,
`RESULTS`, `BUILD`.

In [ ]:
import shutil, sys, tarfile, zipfile
from pathlib import Path

DATA = Path("/content/cache/data")
RESULTS = Path("/content/results")
SEMSEARCH = Path("/content/semsearch")
CACHE = Path("/content/cache")
WAIT_SEC = 1800
EXPECTED = {"train.jsonl", "val_queries.jsonl", "test_queries.jsonl"}


def upload_files_and_verify() -> None:
    # ждёт файлов, если выбор отменили или загрузили раньше старта
    import time
    from google.colab import files

    deadline = time.monotonic() + WAIT_SEC
    while True:
        uploaded = files.upload()
        if uploaded:
            return
        assert time.monotonic() < deadline, "нет загруженных файлов"
        print("файлы не выбраны — откройте диалог и выберите оба файла")

### 3.1. Распаковка архивов

`nllb_translation_results.zip` (если загружен) восстанавливает `cache/`, `results/` и
`semsearch/` в `/content/`; `csn_prepared.tar.gz` — jsonl в `/content/cache/data/`;
`semsearch.zip` — код в `/content/semsearch/`. Архивы удаляются после распаковки.

In [ ]:
def unpack(name: str, target: Path, keep: tuple[str, ...] = ()) -> None:
    # zip: только члены с заданными префиксами верхнего уровня

    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as archive:
            members = [m for m in archive.namelist()
                       if not keep or m.split("/")[0] in keep]
            archive.extractall("/content", members=members)
    else:
        with tarfile.open(name) as archive:
            archive.extractall(target)
    Path(name).unlink()
    print(f"{name}: распакован")


def find_archive(suffix: str) -> str | None:
    # диалог может сохранить архив с другим именем
    matches = sorted(Path.cwd().glob(f"*{suffix}"))
    return str(matches[0]) if matches else None


CACHE.mkdir(parents=True, exist_ok=True)
if not any(CACHE.glob("data/*.jsonl")):
    if (previous := find_archive("results.zip")) is not None:
        unpack(previous, Path("/content"), keep=("cache", "results", "semsearch"))
    if (uploaded := find_archive("csn_prepared.tar.gz")) is not None:
        unpack(uploaded, DATA)
else:
    print("данные уже на месте")

### 3.2. Код пакета и проверка путей

In [ ]:
if not SEMSEARCH.exists():
    if (archive := find_archive("semsearch.zip")) is not None:
        unpack(archive, Path("/content"))
    else:
        upload_files_and_verify()          # выбрали semsearch.zip и csn_prepared.tar.gz
        unpack(find_archive("semsearch.zip"), Path("/content"))

missing = EXPECTED - {item.name for item in DATA.glob("*.jsonl")}
assert not missing, f"в {DATA} нет {sorted(missing)} — загрузите csn_prepared.tar.gz"
sys.path.insert(0, "/content")
import semsearch
print("код:", semsearch.__file__)
print("данные:", sorted(item.name for item in DATA.glob("*.jsonl")))

## 4. Инициализация переводчика (замер времени загрузки)

`model_for()` строит ровно тот `Seq2SeqBackend`, что создаёт `translate.py`, но кэширует
его: NLLB — одна модель на два направления, в памяти нужна одна копия.
`translate()` подменяет на время вызова конструктор `Seq2SeqBackend` в модуле на готовый
бэкенд — `translate_file` дальше работает без изменений и пишет полную сводку
(модель, поле, число переведённых/из кэша, время загрузки, время перевода, запр./с).

In [ ]:
import os, time

os.environ["SEMSEARCH_HOME"] = "/content/cache"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

import torch
import semsearch.data.translate as T
from semsearch.config import TranslateConfig, cache_dir, data_dir
from semsearch.data.translate import DIRECTIONS, Seq2SeqBackend

CFG = TranslateConfig(batch_size=BATCH_SIZE, num_beams=NUM_BEAMS)
LOAD: dict[str, tuple[Seq2SeqBackend, float]] = {}   # cache_id -> (модель, время загрузки)
SUMMARY: list[dict] = []                             # сводки translate_file, по одной на файл


def model_for(direction: str) -> Seq2SeqBackend:
    # модель загружается один раз на пару (модель, языки)
    spec = DIRECTIONS[direction]
    if spec.cache_id not in LOAD:
        torch.cuda.empty_cache()
        started = time.perf_counter()
        backend = Seq2SeqBackend(spec.name, device=DEVICE, batch_size=CFG.batch_size,
                                 num_beams=CFG.num_beams,
                                 max_input_tokens=CFG.max_input_tokens,
                                 max_new_tokens=CFG.max_new_tokens,
                                 src_lang=spec.src_lang, tgt_lang=spec.tgt_lang)
        LOAD[spec.cache_id] = (backend, time.perf_counter() - started)
        print(f"{spec.cache_id}: загрузка {LOAD[spec.cache_id][1]:.1f} с")
    return LOAD[spec.cache_id][0]


def translate(name: str, field: str, out_field: str, direction: str) -> None:
    # вход data/<name>.jsonl, выход /content/results/<name>.jsonl
    RESULTS.mkdir(parents=True, exist_ok=True)
    backend = model_for(direction)
    original = T.Seq2SeqBackend
    T.Seq2SeqBackend = lambda *args, **kwargs: backend
    try:
        summary = T.translate_file(
            input_path=DATA / name, output_path=RESULTS / name,
            field=field, out_field=out_field, spec=DIRECTIONS[direction],
            limit=None, device=DEVICE, cfg=CFG)
    finally:
        T.Seq2SeqBackend = original
    summary["out_field"] = out_field
    summary["device"] = backend.device   # бэкенд мог откатиться на CPU
    print(f"{name} → {out_field}: {summary['translated_by_model']} моделью, "
          f"{summary['from_cache']} из кэша за {summary['elapsed_sec']:.0f} с "
          f"({summary['sentences_per_sec']} запр./с)")
    SUMMARY.append(summary)


print("cache:", cache_dir())
print("данные:", data_dir())

## 5. train EN→RU (100 000 текстов) → поле `query_ru`

In [ ]:
translate("train.jsonl", field="query_en", out_field="query_ru", direction="nllb-en2ru")

## 6. train RU→EN (100 000 текстов) → поле `query_en_back`

Обратный перевод по нашим же `query_ru` (круговой перевод; предупреждение о смещении
для BM25 — `DECISIONS.md` §8).

In [ ]:
translate("train.jsonl", field="query_ru", out_field="query_en_back", direction="nllb-ru2en")

## 7. val_queries (2 000) и test_queries (1 000): EN→RU и RU→EN

In [ ]:
for split in ("val_queries.jsonl", "test_queries.jsonl"):
    translate(split, field="query_en", out_field="query_ru", direction="nllb-en2ru")

In [ ]:
for split in ("val_queries.jsonl", "test_queries.jsonl"):
    translate(split, field="query_ru", out_field="query_en_back", direction="nllb-ru2en")

## 8. Отчёт: время, скорость, размеры файлов

Пишется `/content/results/translation_report.json` (сводки `translate_file` + время
загрузки модели) и печатается таблица. Для круглого прогона без кэша время одной
стороны ≈ 100 000 / (запр./с из строки train).

In [ ]:
import json
from datetime import datetime

report = {
    "gpu": torch.cuda.get_device_name(0),
    "batch_size": CFG.batch_size,
    "num_beams": CFG.num_beams,
    "model_load_sec": {key: round(sec, 1) for key, (_, sec) in LOAD.items()},
    "total_elapsed_sec": round(sum(item["elapsed_sec"] for item in SUMMARY), 0),
    "files": SUMMARY,
}
(RESULTS / "translation_report.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"GPU {report['gpu']}, beams={CFG.num_beams}, batch={CFG.batch_size}")
for key, sec in report["model_load_sec"].items():
    print(f"загрузка {key}: {sec} с")
print(f"\n{'файл':<20}{'поле':<16}{'N':>7}{'кэш':>8}{'моделью':>9}{'с':>8}{'запр./с':>10}")
for item in SUMMARY:
    print(f"{Path(item['input']).name:<20}{item['out_field']:<16}"
          f"{item['sentences']:>7}{item['from_cache']:>8}{item['translated_by_model']:>9}"
          f"{item['elapsed_sec']:>8.0f}{item['sentences_per_sec']:>10.2f}")
print(f"\nвсего перевода: {report['total_elapsed_sec'] / 60:.0f} мин")
print(f"{'файл':<24}{'МБ':>8}")
for path in sorted(RESULTS.glob("*.json*")):
    print(f"{path.name:<24}{path.stat().st_size / 1024**2:>8.1f}")

## 9. Сохранить результаты (можно запускать вручную в любой момент)

Архив `nllb_translation_results.zip` в `/content/`: `cache/` (кэш переводов),
`results/` (jsonl + отчёт), `semsearch/` (код, чтобы архив был самодостаточным).
Скачивание на канал T4 может занять пару минут (архив ~150–200 МБ).
Важные ограничения: jsonl содержат исходный код функций, поэтому архив не выкладывается
в публичные места; кэш ускоряет только повторный перевод, к обучению он не нужен.

In [ ]:
from google.colab import files
import shutil

archive = Path("/content/nllb_translation_results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as handle:
    for folder in (CACHE, RESULTS, SEMSEARCH):
        for path in folder.rglob("*"):
            if path.is_file() and "__pycache__" not in path.parts:
                handle.write(path, path.relative_to("/content"))
print(f"{archive.name}: {archive.stat().st_size / 1024**2:.1f} МБ")
files.download(str(archive))
print("сохранено:", datetime.now().strftime("%Y-%m-%d %H:%M"))